<div class="alert alert-block alert-info" style="margin-top: 20px">
    <center>
    <h1 align=center><font size = 5>Dual Moving Bubble Charts<br><br>Time Use in Canada and Vietnam<br><br>By Thuc Dao (Thomas Dao)</font></h1>
    </center>
</div>

### Install libraries (run once)

Until July 2025, the d3blocks package is not available in the conda-forge or default Anaconda channels for Windows OS.  
Thus, install d3blocks using pip (not conda) in a new Conda environment.

Run the following cell in the __Anaconda command prompt__.

In [ ]:
conda create -n d3blocks_env
conda activate d3blocks_env

#conda config --env --add channels conda-forge
#conda config --env --set channel_priority strict

#conda install d3blocks
pip install d3blocks

# Install a dependency of d3blocks:
# EITHER charset_normalizer (preferred, as it is more modern)
#conda install charset_normalizer
pip install charset_normalizer

# OR chardet (not recommended)
#conda install chardet
#pip install chardet

conda install jupyter notebook
conda install ipykernel
python -m ipykernel install --name d3blocks_env --display-name "d3blocks_env"
conda activate d3blocks_env
jupyter notebook

Remember to choose the d3blocks_env environment in order to use the d3blocks library.

If you forget it, you can also change the environment inside the Jupyter Notebook, by selecting in the menu:  
Kernel >>> Change Kernel >>> d3blocks_env

Note: Do NOT use sas7bdat Python package to read .sas7bdat files.  
Reason: It only works with uncompressed .sas7bdat files and will fail on modern compressed files from SAS 9.4+ (like most StatCan data).

Run the following cells in the __Anaconda command prompt__.

In [ ]:
conda activate d3blocks_env
conda install -c conda-forge pyreadstat

### Import libraries

In [31]:
import pyreadstat
import os
import shutil
import pandas as pd
from d3blocks import D3Blocks
from datetime import datetime
from time import gmtime, strftime

### Helper Functions

In [ ]:
def convert_time(minute):
    return strftime("%d-%m-%Y %H:%M:%S", gmtime(int(minute) * 60))

In [ ]:
def summarize_activity_data(df):
    """
    Prints summary statistics and top activities from the provided DataFrame.
    """
    print("Total records:", len(df))
    print("Unique individuals:", df['sample_id'].nunique())
    print("Time range:", df['datetime'].min(), "to", df['datetime'].max())
    print("\nTop 10 activities by frequency:")
    print(df['state'].value_counts().head(10))

In [ ]:
def get_safe_filepath(filepath):
    """
    Checks if a file exists. If it does, prompts the user to overwrite or rename.
    """
    while os.path.exists(filepath):
        print(f"\n[!] Warning: The file '{filepath}' already exists.")
        choice = input("Do you want to (o)verwrite it or (r)ename the new file? [o/r]: ").strip().lower()
        
        if choice == 'o':
            print(f"Overwriting '{filepath}'...")
            break # Breaks loop, returns the original filepath to overwrite
        elif choice == 'r':
            new_name = input("Enter the new file name: ").strip()
            # Ensure the new file still goes into the correct directory
            directory = os.path.dirname(filepath)
            filepath = os.path.join(directory, new_name)
            # The loop continues to check if this NEW name also exists
        else:
            print("Invalid choice. Please enter 'o' to overwrite or 'r' to rename.")
            
    return filepath

In [ ]:
def get_vn_activity(code):
    code = int(code)
    if code in [101, 102, 199, 201, 202, 299, 301, 302, 399, 401, 402, 499, 502]:
        return "Work"
    elif code in [198, 298, 298, 498, 598, 698, 798, 898, 998, 1098, 1198, 1298, 1398, 1498, 1598]:
        return "Travelling"
    elif code in [501]:
        return "Sell food"
    elif code in [504, 505, 506, 507, 508]:
        return "Provide services"
    elif code in [601]:
        return "Housework"
    elif code in [602]:
        return "Shopping"
    elif code in [701, 702]:
        return "Caring"
    elif code in [901, 902, 903]:
        return "Education"
    elif code in [1201, 1202, 1203, 1299]:
        return "Entertainment"
    elif code in [1301, 1302, 1399]:
        return "Sport"
    elif code == 1402:
        return "TV/Youtube"
    elif code == 1404:
        return "Surf web"
    elif code == 1501:
        return "Sleeping"
    elif code == 1502:
        return "Eating"
    elif code == 1503:
        return "Personal hygiene"
    elif code == 1506:
        return "Relaxing"
    else:
        return "Others"

In [ ]:
# Define the universal order for Canada and Vietnam
category_order = {
    "Sleeping": "01. Sleeping",
    "Personal hygiene": "02. Personal hygiene",
    "Eating": "03. Eating",
    "Housework": "04. Housework",
    "Caring": "05. Caring",
    "Work": "06. Work",
    "Education": "07. Education",
    "Provide services": "08. Provide services",
    "Sell food": "09. Sell food",
    "Shopping": "10. Shopping",
    "Travelling": "Travelling",
    "Sport": "11. Sport",
    "Relaxing": "12. Relaxing",
    "Entertainment": "13. Entertainment",
    "TV/Youtube": "14. TV/Youtube",
    "Surf web": "15. Surf web",
    "Others": "16. Others"
}

### Load Vietnamese data

In [ ]:
# Input path
vn_file = r'input\VN_time_use.csv'

print("Loading Vietnamese data from CSV file...")
df_vn = pd.read_csv(vn_file, usecols=["ID", "BEGIN", "Q401"],
                   encoding='latin-1',
                   converters={"Q401": get_vn_activity, "BEGIN": convert_time})

print("✅ VN data loaded.")

### Process and transform Vietnamese time use data

In [ ]:
print("Processing Vietnamese data...")
df_vn = df_vn.sort_values(["ID", "BEGIN"])
df_vn = df_vn.rename(columns={"ID": "sample_id", "BEGIN": "datetime", "Q401": "state"})
df_vn = df_vn.dropna(subset=['sample_id', 'datetime', 'state'])
df_vn['datetime'] = pd.to_datetime(df_vn['datetime'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
df_vn = df_vn.dropna(subset=['datetime'])

# Apply the category order
df_vn['state'] = df_vn['state'].map(category_order)

summarize_activity_data(df_vn)

### Generate a Vietnamese moving bubble chart

In [ ]:
# Create the 'output' folder if it doesn't already exist
os.makedirs("output", exist_ok=True)

# Check file existence for VN chart
vn_chart_path = get_safe_filepath("output/VN_moving_bubble_chart.html")

# Generate Vietnamese Moving Bubble Chart
d3_vn = D3Blocks()
d3_vn.movingbubbles(
                    df_vn,
                    filepath=vn_chart_path,
                    title='Daily Time Use in Vietnam',
                    note='',
                    figsize=(550, 600),
                    size=1,
                    datetime='datetime',
                    sample_id='sample_id',
                    state='state',
                    center='Travelling',
                    cmap='hsv',
                    save_button=True # the SVG save button
)

### Map Canadian activity codes to Vietnamese-style categories

| Canadian Code(s)          | Description                                        | VN Category Equivalent                |
| ------------------------- | -------------------------------------------------- | ------------------------------------- |
| 501–507, 599              | Paid work, selling, training, looking for work     | **Work**                              |
| 505                       | Selling of goods or services for pay               | **Sell food** or **Provide services** |
| 201–209, 231–241          | Cooking, cleaning, household chores, DIY, repairs  | **Housework**                         |
| 151–154, 404              | Eating or drinking, food-related travel            | **Eating**                            |
| 100–109                   | Sleep, nap, rest                                   | **Sleeping**                          |
| 126–130, 199, 401         | Personal hygiene, medical, personal care           | **Personal hygiene**                  |
| 150                       | Eating/drinking (top category)                     | **Eating**                            |
| 260–264                   | In-person or online shopping, research             | **Shopping**                          |
| 300–307, 350–353, 801–802 | Caring for children/adults                         | **Caring**                            |
| 601–604, 600              | School, homework, learning, tutoring               | **Education**                         |
| 1001–1005, 413            | Exercise, sport, physical activity, related travel | **Sport**                             |
| 1202                      | Watching TV, movies, online videos                 | **TV/Youtube**                        |
| 1201, 1203–1204           | Reading, music, podcasts, general tech use         | **Surf web** or **Entertainment**     |
| 1105                      | Arts, hobbies, games, crafts, video games          | **Entertainment**                     |
| 1106                      | Outdoor leisure, hobbies, birdwatching, boating    | **Relaxing** or **Entertainment**     |
| 1301–1303                 | Waiting, doing nothing, thinking                   | **Relaxing** or **Others**            |
| 409, 407, 411, 400–416    | Travel for work, shopping, study, care, etc.       | **Travelling**                        |
| 9999, 1304                | Unspecified or other activities                    | **Others**                            |


**Notes:**  
Categories like “Sell food” or “Provide services” don’t map directly from Canadian codes  
but can be inferred from specific self-employment or home-business entries (505).  
These can be differentiated only with further context.

In [ ]:
ca_to_vn = {
    "Work": [501, 502, 503, 504, 505, 506, 507, 599],
    "Sell food": [505],
    "Provide services": [505],
    "Housework": list(range(201, 210)) + list(range(231, 242)),
    "Eating": [150, 151, 152, 153, 154, 404],
    "Sleeping": list(range(100, 110)),
    "Personal hygiene": [126, 127, 128, 129, 130, 199, 401],
    "Shopping": [260, 261, 262, 263, 264],
    "Caring": list(range(300, 308)) + list(range(350, 354)) + [801, 802],
    "Education": [600, 601, 602, 603, 604],
    "Sport": [1001, 1002, 1003, 1004, 1005, 413],
    "TV/Youtube": [1202],
    "Surf web": [1201, 1203, 1204],
    "Entertainment": [1105, 1106],
    "Relaxing": [1301, 1302, 1303],
    "Travelling": list(range(400, 417)),
    "Others": [9999, 1304]
}

In [ ]:
def map_ca_activity(code):
    try:
        if pd.isna(code): return "Others"
        code = int(float(code)) # Handle float codes
        for category, codes in ca_to_vn.items():
            if code in codes:
                return category
    except (ValueError, TypeError):
        pass
    return "Others"

### Load Canadian data with pyreadstat (at least 5x faster than pandas.read_sas)

In [ ]:
# Input path
ca_file = r'Data sources\CAN_TU_ET_2022\Data_Données\TU_ET_2022_Episode_PUMF.sas7bdat'

# Conversion path
#ca_file_csv = r'input\CA_time_use.csv'

print("Loading Canadian data from SAS file...")
#df_ca, meta = pyreadstat.read_sas7bdat(ca_file)
df_ca, meta = pyreadstat.read_sas7bdat(ca_file, usecols=["PUMFID", "STARTMIN", "TUI_01"])

# Save as CSV
#df_ca.to_csv(ca_file_csv, index=False, encoding='utf-8-sig')

print("✅ CA data loaded and saved to CSV")

### Process and transform Canadian time use data

In [ ]:
print("Processing Canadian data...")
#ca_file = r'input\CA_time_use.csv'
#df_ca = pd.read_csv(ca_file)[["PUMFID", "STARTMIN", "TUI_01"]].dropna()

# Standardize column values
df_ca["datetime"] = df_ca["STARTMIN"].apply(convert_time)
df_ca["state"] = df_ca["TUI_01"].apply(map_ca_activity)
df_ca = df_ca.rename(columns={"PUMFID": "sample_id"}).sort_values(["sample_id", "datetime"])
df_ca = df_ca.dropna(subset=['sample_id', 'datetime', 'state'])

# Apply the category order
df_ca['state'] = df_ca['state'].map(category_order)

summarize_activity_data(df_ca)

### Generate a Canadian moving bubble chart

In [ ]:
# Check file existence for CA chart
ca_chart_path = get_safe_filepath("output/CA_moving_bubble_chart.html")

# Generate Canadian Moving Bubble Chart
d3_ca = D3Blocks()
d3_ca.movingbubbles(
                    df_ca,
                    filepath=ca_chart_path,
                    title='Daily Time Use in Canada',
                    note='',
                    figsize=(550, 600),
                    size=1,
                    datetime='datetime',
                    sample_id='sample_id',
                    state='state',
                    center='Travelling',
                    cmap='hsv',
                    save_button=True # the SVG save button
)

### Pack two moving bubble charts together with responsive layout

In [ ]:
print("Generating dual view layout...")

# Extract JUST the filenames (without the 'output/' folder path) so the HTML iframe can load them
vn_filename = os.path.basename(vn_chart_path)
ca_filename = os.path.basename(ca_chart_path)

# Check file existence for the Dual chart
dual_filepath = get_safe_filepath("output/Dual_moving_bubble_chart.html")

# Notice the 'f' before the string quotes. This allows us to inject {ca_filename} dynamically.
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Time User in Canada and Vietnam</title>
<style>
  body {
      font-family: Arial, sans-serif;
      margin: 0;
      padding: 0;
      background-color: #f4f4f9;
  }
  h2 {
      text-align: center;
      padding: 20px 0;
      margin: 0;
      color: #333;
  }
  .container {
      display: flex;
      flex-wrap: wrap;
      justify-content: center; /* Centers the charts */
      width: 100%;
      /* 90vh makes it fill most of the screen vertically */
      height: 90vh; 
      gap: 10px; /* Adds a little breathing room between them */
  }
  .chart-box {
      flex: 1 1 45%; /* Slightly less than 50% to prevent tight squeezing */
      min-width: 550px; /* Matches the new figsize width */
      height: 650px; /* Matches the new figsize height + some padding */
  }
  iframe {
      width: 100%;
      height: 100%;
      border: none;
      overflow: hidden; /* Hides any remaining scrollbars */
  }
  /* Media query to stack the charts vertically on smaller screens (Mobile) */
  @media (max-width: 768px) {
      .chart-box {
          flex: 1 1 100%;
          height: 50vh;
      }
  }
</style>
</head>
<body>
  <h2>Time Use Comparison: Canada vs Vietnam</h2>
  <div class="container">
      <div class="chart-box">
          <iframe src="{ca_filename}" title="Canadian Time Use"></iframe>
      </div>
      <div class="chart-box">
          <iframe src="{vn_filename}" title="Vietnamese Time Use"></iframe>
      </div>
  </div>
</body>
</html>
"""

# Write the HTML wrapper to your directory
with open(dual_filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Process completed! You can now open '{dual_filepath}' in your browser.")